In [2]:
import os
import math
import warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    MobileBertModel,
    MobileBertPreTrainedModel,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. MODELING CLASS: STUDENT (1-LOGIT OUTPUT)
# ==============================================================================
class MobileBertForSequenceClassificationBCE(MobileBertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = 1

        self.mobilebert = MobileBertModel(config)

        drop_prob = getattr(config, "classifier_dropout", None)
        classifier_dropout = drop_prob if drop_prob is not None else 0.1

        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, self.num_labels)

        nn.init.xavier_normal_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        outputs = self.mobilebert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )

        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)

        logits = self.classifier(pooled_output).squeeze(-1)
        return logits

# ==============================================================================
# 2. MAIN: KNOWLEDGE DISTILLATION (TEACHER -> STUDENT)
# ==============================================================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Using device: {device}")

    # --- A. NẠP TEACHER MODEL (Pretrained HF) ---
    TEACHER_ID = "distilbert-base-uncased-finetuned-sst-2-english"
    print(f"[INFO] Loading Pretrained Teacher: {TEACHER_ID}...")
    teacher_model = AutoModelForSequenceClassification.from_pretrained(TEACHER_ID).to(device)
    teacher_model.eval() # Khóa Teacher ở chế độ đánh giá

    # --- B. KHỞI TẠO STUDENT MODEL (Train từ đầu) ---
    STUDENT_ID = "google/mobilebert-uncased"
    print(f"[INFO] Initializing Student ({STUDENT_ID}) FROM SCRATCH...")
    # Chỉ nạp Config, KHÔNG nạp from_pretrained để khởi tạo trọng số ngẫu nhiên hoàn toàn
    student_config = AutoConfig.from_pretrained(STUDENT_ID)
    student_model = MobileBertForSequenceClassificationBCE(student_config).to(device)

    # --- C. CHUẨN BỊ DỮ LIỆU (SST-2 với namespace chuẩn nyu-mll) ---
    tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)
    print("[INFO] Loading SST-2 Dataset...")
    dataset = load_dataset("nyu-mll/glue", "sst2")

    train_data = dataset["train"].shuffle(seed=42)
    eval_data = dataset["validation"]

    def preprocess_function(examples):
        tokenized = tokenizer(examples["sentence"], truncation=True, max_length=128)
        tokenized["labels"] = [float(l) for l in examples["label"]]
        return tokenized

    print("[INFO] Tokenizing dataset...")
    tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=["sentence", "label", "idx"])
    tokenized_eval = eval_data.map(preprocess_function, batched=True, remove_columns=["sentence", "label", "idx"])

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_loader = DataLoader(tokenized_train, batch_size=32, shuffle=True, collate_fn=data_collator)
    eval_loader = DataLoader(tokenized_eval, batch_size=32, shuffle=False, collate_fn=data_collator)

    # --- D. CẤU HÌNH OPTIMIZER & DISTILLATION LOSS ---
    criterion = nn.BCEWithLogitsLoss()
    ALPHA = 0.5  # Trọng số cân bằng giữa Hard Loss (Label) và Soft Loss (Teacher)

    optimizer = torch.optim.AdamW(student_model.parameters(), lr=5e-5, weight_decay=0.01)

    EPOCHS = 3
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=50, num_training_steps=total_steps)

    # --- E. VÒNG LẶP HUẤN LUYỆN ---
    print("\n" + "="*85)
    print(f"{'Epoch':<8}{'Train Loss (Total)':<22}{'Val Loss':<15}{'Accuracy':<14}{'F1':<10}")
    print("="*85)

    best_f1 = 0.0
    OUTPUT_DIR = "./student_sst2_distilled"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        # 1. TRAIN PHASE
        student_model.train()
        running_train_loss = 0.0

        for step, batch in enumerate(train_loader):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch.get("token_type_ids", None)
            labels = batch["labels"].to(device)

            # Lấy xác suất dự đoán từ Teacher (Không tính đạo hàm)
            with torch.no_grad():
                teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
                teacher_probs = torch.softmax(teacher_outputs.logits, dim=-1)[:, 1]

            # Forward Student
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)
            student_logits = student_model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

            # -------------------------------------------------------------
            # DISTILLATION LOSS (BCE)
            # -------------------------------------------------------------
            loss_hard = criterion(student_logits, labels)
            loss_soft = criterion(student_logits, teacher_probs)

            # Tổng hợp Loss
            loss = (1.0 - ALPHA) * loss_hard + ALPHA * loss_soft

            loss.backward()
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            running_train_loss += loss.item() * labels.size(0)

        epoch_train_loss = running_train_loss / len(train_data)

        # 2. EVALUATION PHASE
        student_model.eval()
        running_eval_loss = 0.0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in eval_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                token_type_ids = batch.get("token_type_ids", None)
                if token_type_ids is not None:
                    token_type_ids = token_type_ids.to(device)

                labels = batch["labels"].to(device)

                student_logits = student_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )
                loss = criterion(student_logits, labels)
                running_eval_loss += loss.item() * labels.size(0)

                probs = torch.sigmoid(student_logits)
                preds = (probs >= 0.5).long().cpu().numpy()

                all_preds.extend(preds)
                all_labels.extend(labels.long().cpu().numpy())

        epoch_eval_loss = running_eval_loss / len(eval_data)
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average="weighted")

        print(f"{epoch:<8}{epoch_train_loss:<22.6f}{epoch_eval_loss:<15.6f}{acc:<14.6f}{f1:<10.6f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(student_model.state_dict(), f"{OUTPUT_DIR}/best_model.pt")

    tokenizer.save_pretrained(OUTPUT_DIR)
    print("="*85)
    print(f"[INFO] Done! Best model saved to {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

[INFO] Using device: cuda
[INFO] Loading Pretrained Teacher: distilbert-base-uncased-finetuned-sst-2-english...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[INFO] Initializing Student (google/mobilebert-uncased) FROM SCRATCH...
[INFO] Loading SST-2 Dataset...


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

...

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

[INFO] Tokenizing dataset...


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]


Epoch   Train Loss (Total)    Val Loss       Accuracy      F1        
1       0.403464              0.480117       0.803899      0.803802  
2       0.227274              0.485116       0.802752      0.802503  
3       0.182136              0.548230       0.799312      0.799252  
[INFO] Done! Best model saved to ./student_sst2_distilled
